In [1]:
'''Python 測驗 task 3 參考解答'''


import json


def analyze_gcp_logs(filename: str, service: str = None) -> None:
    '''analyze the GCP SPEC log file and aggregate the servities of services
    filename - log file name in JSON
    service  - specific service name to be filtered
    return   - None

    thoughts - 1. 設計 log 資料以 dict 儲存，才能達到動態增加 service_name 與 severity 的彈性
                  ex: services = {'pauca': {'INFO': 1}, 'wt': {'NOTICE': 1}, ...}
               2. 為求效率，for 迴圈掃描 log 只能進行一次，同時只處理參數指定的 service_name,
                  若未指定 service_name 則所有 service_name 皆保留。
               3. 輸出時才依據各 service_name 對應到的 serverity 名稱與數量計算須顯示的最大長度，
                  順道依據名稱排序，達成輸出可讀性。
               4. 防呆並強化 function，避免輸入錯誤導致 function 執行報錯，包含開檔失敗的錯誤攔截
                  以及 log 格式缺失過濾。
                  a. 開檔錯誤以 try-except 攔截，被攔截即結束整個 function 但加入提示輸出
                  b. log 格式以正面濾除方式處理，故須善用 dict.get(, {}) 而非 dict subscription，
                     ex: services['wt']。格式缺失的 log 即忽略。
    '''
    try:
        with open(filename, encoding='utf-8') as f:
            logs = json.load(f)
    except:
        print('bad log file')
        return
    
    services = {}  # to keep severity info of services, ex: {'pauca': {'INFO': 1}, 'wt': {'NOTICE': 1}, ...}
    
    # count severities of services
    for log in logs:    
        service_name = log.get('resource', {}).get('labels', {}).get('service_name')  # get default {} may ease the error handling
        severity = log.get('severity')
    
        # fill in service severity info if both service_name and severity exist, and service_name matches the assigned one
        if service_name and severity and (service == None or service == service_name):
            if service_name in services:
                services[service_name][severity] = services[service_name].get(severity, 0) + 1
            else:
                services[service_name] = {severity: 1}
        
    # output, 短但是慢的寫法
    # if services:
    #     severities = sorted({serverity for service_value in services.values() for serverity in service_value})  # sorted severities
    #     severity_lens = [max(len(str(service_value.get(serverity, 0))) for service_value in services.values()) for serverity in severities]  # max serverity value lens
    #
    #     service_len = max(len(service_name) for service_name in services)  # max service name len
    #     for service_name in services:
    #         counts = [f'{serverity}: {services[service_name].get(serverity, 0):{severity_lens[i]}}' for i, serverity in enumerate(severities)]  # part of serverity counts to be showed
    #         print(f'{service_name:{service_len}} {"/ ".join(counts)}')

    # output, 長但是快的寫法
    if services:
        # 同時取得 serverity 與佔用位數對應
        severities_format = {}
        for s in services.values():
            for serverity, count in s.items():
                len_count = len(str(count))
                if serverity not in severities_format or severities_format[serverity] < len_count:
                    severities_format[serverity] = len_count
        severities_format = sorted(severities_format.items())  # sorted severities
        
        service_len = max(len(service) for service in services)  # max service name len
        for service in services:
            counts = [f'{serverity}: {services[service].get(serverity, 0):{s}}' for serverity, s in severities_format]  # part of serverity counts to be showed
            print(f'{service:{service_len}} {"/ ".join(counts)}')
    
    else:
        print('no log be filtered')

In [2]:
analyze_gcp_logs('serviceslogs.json')

pauca ERROR: 26/ INFO: 23/ NOTICE: 0/ WARNING: 2
myca  ERROR:  0/ INFO:  3/ NOTICE: 0/ WARNING: 0
wt    ERROR: 36/ INFO: 53/ NOTICE: 1/ WARNING: 0


In [3]:
analyze_gcp_logs('hello.json')

bad log file


In [4]:
analyze_gcp_logs('serviceslogs.json', 'wt')

wt ERROR: 36/ INFO: 53/ NOTICE: 1


In [5]:
analyze_gcp_logs('serviceslogs.json', 'test')

no log be filtered
